In [ ]:
# ============================================================================
# ROOT-CAUSE ANALYSIS  (STEP 1 of 2) — setup & load evidence
#
# For each deviation we ask an LLM to read the highest-level generated context
# (contextual_retrieval_text_full = deterministic refs + llm context + source)
# alongside the raw deviation free text (source_free_text_full) and judge the
# most likely ROOT CAUSE.
#
# Served via the Databricks Foundation Model API (OpenAI-compatible) using the
# notebook's own workspace token — no external API key, same as the glossary.
# ============================================================================
import time, json, re, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from openai import OpenAI
from pyspark.sql import functions as F, types as T

CATALOG = "us_gmsgq_dev"
ALYT    = "gms_us_alyt"

EMBED_INPUT = f"{CATALOG}.{ALYT}.deviation_embed_input"       # per pr_id
RC_TABLE    = f"{CATALOG}.{ALYT}.deviation_root_cause"        # output

MODEL_NAME  = "databricks-gpt-5-mini"   # swap to databricks-meta-llama-3-3-70b-instruct if preferred
PARALLEL    = 5                         # tune down if you hit endpoint rate limits
LIMIT_N     = None                      # set to e.g. 20 to smoke-test on a few events first

# Databricks Foundation Model APIs are OpenAI-compatible. Authenticate with the
# notebook's own workspace token and point at the serving-endpoints base URL.
DBX_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
DBX_HOST  = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
client = OpenAI(api_key=DBX_TOKEN, base_url=f"{DBX_HOST}/serving-endpoints")

# ---- Load evidence: highest-level context + its raw free text --------------
rc_pdf = (
    spark.table(EMBED_INPUT)
    .select("pr_id", "source_free_text_full", "contextual_retrieval_text_full")
    .toPandas()
)
rc_pdf["pr_id"] = rc_pdf["pr_id"].astype(str)
for c in ["source_free_text_full", "contextual_retrieval_text_full"]:
    rc_pdf[c] = rc_pdf[c].fillna("")

if LIMIT_N:
    rc_pdf = rc_pdf.head(LIMIT_N).copy()

print(f"Loaded {len(rc_pdf):,} deviations for root-cause analysis")

In [ ]:
# ============================================================================
# ROOT-CAUSE ANALYSIS  (STEP 2 of 2) — prompt, judge, and parse
#
# A fixed category taxonomy keeps judgements consistent across events. The LLM
# returns strict JSON so we can store structured columns.
# ============================================================================

# Consistent RCA taxonomy (GMP / clinical deviation style). "Other / Indeterminate"
# is the escape hatch when evidence is insufficient.
ROOT_CAUSE_CATEGORIES = [
    "Human Error",
    "Procedure / SOP Gap",
    "Training / Competency",
    "Equipment / System Failure",
    "Documentation Error",
    "Communication / Handoff",
    "Vendor / Supplier / CRO",
    "Material / Sample Issue",
    "Scheduling / Timing",
    "Process Design",
    "Other / Indeterminate",
]

SYSTEM_RCA_PROMPT = """You are a clinical-trial quality investigator performing root-cause analysis (RCA) on deviation records.
Judge the single MOST LIKELY root cause from the evidence provided. Do not speculate beyond it.
Choose exactly one category from this list:
{categories}

Respond with ONLY a JSON object, no prose, in this exact shape:
{{
  "root_cause_category": "<one category from the list above, verbatim>",
  "root_cause_summary": "<1-3 sentence explanation grounded in the evidence>",
  "contributing_factors": ["<short factor>", "..."],
  "confidence": "high | medium | low"
}}
If the evidence is too thin to judge, use "Other / Indeterminate" with confidence "low"."""

USER_RCA_PROMPT = """<enriched_context>
{context}
</enriched_context>

<raw_deviation_text>
{free_text}
</raw_deviation_text>

Using the evidence above, determine the most likely root cause of this deviation and answer with the JSON object only."""


def _parse_rca_json(raw: str) -> dict:
    """Best-effort parse of the model's JSON reply."""
    fallback = {
        "root_cause_category": "Other / Indeterminate",
        "root_cause_summary": (raw or "").strip()[:1000],
        "contributing_factors": [],
        "confidence": "low",
    }
    if not raw:
        return fallback
    text = raw.strip()
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.IGNORECASE).strip()
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)   # first {...} block
    if not m:
        return fallback
    try:
        obj = json.loads(m.group(0))
    except json.JSONDecodeError:
        return fallback
    cats = obj.get("contributing_factors", [])
    if isinstance(cats, str):
        cats = [cats]
    return {
        "root_cause_category": str(obj.get("root_cause_category", "Other / Indeterminate")).strip(),
        "root_cause_summary": str(obj.get("root_cause_summary", "")).strip(),
        "contributing_factors": [str(x).strip() for x in cats if str(x).strip()],
        "confidence": str(obj.get("confidence", "low")).strip().lower(),
    }


def judge_root_cause(context: str, free_text: str) -> dict:
    """Ask the LLM for the root cause of one deviation; returns parsed dict + raw."""
    evidence = (context or "").strip() or (free_text or "").strip()
    if not evidence:
        return {**_parse_rca_json(""), "raw_response": ""}

    messages = [
        {"role": "system",
         "content": SYSTEM_RCA_PROMPT.format(categories="\n".join(f"- {c}" for c in ROOT_CAUSE_CATEGORIES))},
        {"role": "user",
         "content": USER_RCA_PROMPT.format(context=context, free_text=free_text)},
    ]
    # NOTE: temperature omitted — the gpt-5 family reasoning endpoints only accept
    # the default. Swap MODEL_NAME to a llama endpoint if you want temperature=0.0.
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        max_tokens=500,
        messages=messages,
    )
    raw = (resp.choices[0].message.content or "").strip()
    return {**_parse_rca_json(raw), "raw_response": raw}

In [ ]:
# ============================================================================
# ROOT-CAUSE ANALYSIS — run in parallel, assemble, and write the table
# Output: us_gmsgq_dev.gms_us_alyt.deviation_root_cause  (one row per pr_id)
# ============================================================================
results = [None] * len(rc_pdf)
lock = threading.Lock()
done = 0

def _work(i, ctx, free):
    return i, judge_root_cause(ctx, free)

t0 = time.time()
with ThreadPoolExecutor(max_workers=PARALLEL) as ex:
    futs = [
        ex.submit(_work, i,
                  rc_pdf["contextual_retrieval_text_full"].iat[i],
                  rc_pdf["source_free_text_full"].iat[i])
        for i in range(len(rc_pdf))
    ]
    for f in as_completed(futs):
        i, judgement = f.result()
        results[i] = judgement
        with lock:
            done += 1
            if done % 100 == 0:
                print(f"  {done:,}/{len(rc_pdf):,} judged ({time.time()-t0:.0f}s)")

print(f"Judged {len(rc_pdf):,} deviations in {time.time()-t0:.0f}s")

# ---- Assemble ---------------------------------------------------------------
rc_out = pd.DataFrame({
    "pr_id":                 rc_pdf["pr_id"].values,
    "root_cause_category":   [r["root_cause_category"]   for r in results],
    "root_cause_summary":    [r["root_cause_summary"]    for r in results],
    "contributing_factors":  [r["contributing_factors"]  for r in results],
    "confidence":            [r["confidence"]            for r in results],
    "raw_response":          [r["raw_response"]          for r in results],
})

print("\nRoot-cause category distribution:")
print(rc_out["root_cause_category"].value_counts().to_string())

# ---- Write (one row per pr_id) ---------------------------------------------
rc_schema = T.StructType([
    T.StructField("pr_id",                T.StringType(),              False),
    T.StructField("root_cause_category",  T.StringType(),              True),
    T.StructField("root_cause_summary",   T.StringType(),              True),
    T.StructField("contributing_factors", T.ArrayType(T.StringType()), True),
    T.StructField("confidence",           T.StringType(),              True),
    T.StructField("raw_response",         T.StringType(),              True),
])
rc_rows = [
    (r.pr_id, r.root_cause_category, r.root_cause_summary,
     list(r.contributing_factors), r.confidence, r.raw_response)
    for r in rc_out.itertuples(index=False)
]
rc_sdf = spark.createDataFrame(rc_rows, schema=rc_schema)
(rc_sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(RC_TABLE))

print(f"\n✓ {RC_TABLE}: {rc_sdf.count():,} rows (one root-cause judgement per pr_id)")
display(rc_sdf.limit(10))